In [1]:
def predict_single_decision(model, file_path, ticker_date, holding_period=15, threshold=0.5):
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler

    def load_excel_data(file_path):
        df = pd.read_excel(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        df = df.sort_index()
        df['Return'] = df['Close'].pct_change()
        df['Volatility'] = df['Return'].rolling(10).std()
        df['Volume_Change'] = df['Volume'].pct_change()
        df.dropna(inplace=True)
        return df

    def compute_tsi(close, r=25, s=13):
        momentum = close.diff()
        abs_momentum = momentum.abs()
        ema1 = momentum.ewm(span=r, min_periods=1).mean()
        ema2 = ema1.ewm(span=s, min_periods=1).mean()
        abs_ema1 = abs_momentum.ewm(span=r, min_periods=1).mean()
        abs_ema2 = abs_ema1.ewm(span=s, min_periods=1).mean()
        tsi = 100 * ema2 / abs_ema2
        return tsi

    def compute_rsi(series, period=14):
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        avg_gain = gain.rolling(window=period).mean()
        avg_loss = loss.rolling(window=period).mean()
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    def add_features(data):
        data['TSI'] = compute_tsi(data['Close'])
        data['RSI'] = compute_rsi(data['Close'])
        data.dropna(inplace=True)
        return data

    def create_labels(data, forward_days=15):
        future_return = data['Close'].shift(-forward_days) / data['Close'] - 1
        data['Label'] = (future_return > 0).astype(int)
        return data.dropna()

    data = load_excel_data(file_path)
    data = add_features(data)
    data = create_labels(data, forward_days=holding_period)


    date = pd.to_datetime(ticker_date).date()
    index_dates = data.index.date

    if date not in index_dates:
        raise ValueError(f"Date {date} not found in data (even after stripping time).")

    matched_idx = np.where(index_dates == date)[0][0]
    matched_date = data.index[matched_idx]

    features = ['Return', 'Volatility', 'Volume_Change', 'TSI', 'RSI']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data[features])
    X = pd.DataFrame(X_scaled, index=data.index, columns=features)

    row_features = X.loc[matched_date].values.reshape(1, -1)
    prob = model.predict_proba(row_features)[0, 1]
    return int(prob > threshold)




In [2]:

import yfinance as yf

def prepare_data_from_yahoo(ticker='SPY', start_date='2003-01-01', end_date='2024-12-31'):
    data = yf.download(ticker, start=start_date, end=end_date)
    data.reset_index(inplace=True)
    data = add_features(data)
    data = create_labels(data, forward_days=15)

    features = ['Return', 'Volatility', 'Volume_Change', 'TSI', 'RSI']
    scaler = StandardScaler()
    X = scaler.fit_transform(data[features])
    y = data['Label']
    X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
        X, y, data, test_size=0.2, shuffle=False)
    return X_train, X_test, y_train, y_test, df_test


In [3]:

def run_strategy_from_yahoo_return_all(ticker='SPY', start_date='2003-01-01', end_date='2024-12-31'):
    print(f"Running mean reversion strategy with XGBoost using Yahoo Finance data for: {ticker}")
    X_train, X_test, y_train, y_test, df_test = prepare_data_from_yahoo(ticker, start_date, end_date)
    model = train_xgboost_model(X_train, y_train)
    df_backtest = backtest_model(model, X_test, df_test)
    return model, X_test, df_test, df_backtest


In [5]:
# === Part 2: Technical indicators ===
def compute_tsi(close, r=25, s=13):
    momentum = close.diff()
    abs_momentum = momentum.abs()
    ema1 = momentum.ewm(span=r, min_periods=1).mean()
    ema2 = ema1.ewm(span=s, min_periods=1).mean()
    abs_ema1 = abs_momentum.ewm(span=r, min_periods=1).mean()
    abs_ema2 = abs_ema1.ewm(span=s, min_periods=1).mean()
    tsi = 100 * ema2 / abs_ema2
    return tsi

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def add_features(data):
    data['TSI'] = compute_tsi(data['Close'])
    data['RSI'] = compute_rsi(data['Close'])
    data.dropna(inplace=True)
    return data

# === Part 3: Label creation ===
def create_labels(data, forward_days=15):
    future_return = data['Close'].shift(-forward_days) / data['Close'] - 1
    data['Label'] = (future_return > 0).astype(int)
    return data.dropna()

In [7]:

from sklearn.preprocessing import StandardScaler

In [8]:

ticker_input = input("Input tickers（example: SPY、AAPL、QQQ）：")
model, X_test, df_test, df_backtest = run_strategy_from_yahoo_return_all(ticker_input)
compare_df = compare_strategies(df_backtest)
print(compare_df)
show_strategy_summary(df_backtest)


Input tickers（example: SPY、AAPL、QQQ）：SPY


[*********************100%***********************]  1 of 1 completed

Running mean reversion strategy with XGBoost using Yahoo Finance data for: SPY


KeyError: "['Return' 'Volatility' 'Volume_Change'] not in index"